In [1]:
from transformers import pipeline

In [1]:
import spacy
import crosslingual_coreference

DEVICE = 0 # Number of the GPU, -1 if want to use CPU

# Add coreference resolution model
coref = spacy.load('en_core_web_sm', disable=['ner', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer'])
coref.add_pipe(
    "xx_coref", config={"chunk_size": 2500, "chunk_overlap": 2, "device": DEVICE})

[nltk_data] Downloading package omw-1.4 to D:\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
error loading _jsonnet (this is expected on Windows), treating C:\Users\zzx12\AppData\Local\Temp\tmpir66py6p\config.json as plain json


KeyboardInterrupt: 

In [7]:
from spacy.cli import download

download("en_core_web_sm")

ConnectionError: HTTPSConnectionPool(host='raw.githubusercontent.com', port=443): Max retries exceeded with url: /explosion/spacy-models/master/compatibility.json (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x00000257CA27A2E0>: Failed to establish a new connection: [Errno 11004] getaddrinfo failed'))

In [5]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

def extract_triplets(text):
    triplets = []
    relation, subject, relation, object_ = '', '', '', ''
    text = text.strip()
    current = 'x'
    for token in text.replace("<s>", "").replace("<pad>", "").replace("</s>", "").split():
        if token == "<triplet>":
            current = 't'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
                relation = ''
            subject = ''
        elif token == "<subj>":
            current = 's'
            if relation != '':
                triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
            object_ = ''
        elif token == "<obj>":
            current = 'o'
            relation = ''
        else:
            if current == 't':
                subject += ' ' + token
            elif current == 's':
                object_ += ' ' + token
            elif current == 'o':
                relation += ' ' + token
    if subject != '' and relation != '' and object_ != '':
        triplets.append({'head': subject.strip(), 'type': relation.strip(),'tail': object_.strip()})
    return triplets
model_path=r'D:\gra_research\codes\fake_news_detection\model_files\rebel-large'
# Load model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to('cuda')
gen_kwargs = {
    "max_length": 512,
    "length_penalty": 0,
    "num_beams": 3,
    "num_return_sequences": 1,
}

# Text to extract triplets from
text = 'Punta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.unta Cana is a resort town in the municipality of Higüey, in La Altagracia Province, the easternmost province of the Dominican Republic.'
import spacy

def coreference_resolution(text):
    # Load the spaCy model
    nlp = spacy.load("en_core_web_md")

    # Process the text with spaCy
    doc = nlp(text)

    # Initialize an empty list to store the processed sentences
    processed_sentences = []

    # Iterate through the sentences in the doc
    for sent in doc.sents:
        # Initialize the sentence text with the original text
        new_sent = sent.text

        # Iterate through the mentions in the sentence
        for mention in sent._.coref_clusters:
            # Replace the mention with the referent text
            new_sent = new_sent.replace(mention.mention.text, mention.referent.text)

        # Append the processed sentence to the list
        processed_sentences.append(new_sent)

    # Join the processed sentences and return the final processed text
    return " ".join(processed_sentences)
text=coreference_resolution(text)
# Tokenizer text
model_inputs = tokenizer(text, max_length=256, padding=True, truncation=True, return_tensors = 'pt')

# Generate
generated_tokens = model.generate(
    model_inputs["input_ids"].to(model.device),
    attention_mask=model_inputs["attention_mask"].to(model.device),
    **gen_kwargs,
)
print(model.device)

# Extract text
decoded_preds = tokenizer.batch_decode(generated_tokens, skip_special_tokens=False)

# Extract triplets
for idx, sentence in enumerate(decoded_preds):
    print(f'Prediction triplets sentence {idx}')
    print(extract_triplets(sentence))


AttributeError: [E046] Can't retrieve unregistered extension attribute 'coref_clusters'. Did you forget to call the `set_extension` method?

In [21]:
from fastcoref import FCoref
model=FCoref(device='cuda:0')

Downloading:   0%|          | 0.00/819 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/393 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/780k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/239 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/345M [00:00<?, ?B/s]

01/18/2023 12:16:49 - INFO - 	 missing_keys: []
01/18/2023 12:16:49 - INFO - 	 unexpected_keys: []
01/18/2023 12:16:49 - INFO - 	 mismatched_keys: []
01/18/2023 12:16:49 - INFO - 	 error_msgs: []
01/18/2023 12:16:49 - INFO - 	 Model Parameters: 90.5M, Transformer: 82.1M, Coref head: 8.4M


In [24]:
from fastcoref import spacy_component
import spacy


text = 'Alice goes down the rabbit hole. Where she would discover a new reality beyond her expectations.'

nlp = spacy.load("en_core_web_sm",exclude=["parser", "lemmatizer", "ner", "textcat"])
nlp.add_pipe(
   "fastcoref",
   config={'model_architecture': 'LingMessCoref', 'model_path': 'biu-nlp/lingmess-coref', 'device': 'cuda:0'}
)

doc = nlp(text,component_cfg={"fastcoref": {'resolve_text': True}})
doc._.resolved_text


Downloading:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/361 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/780k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/446k [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.29M [00:00<?, ?B/s]

...

01/18/2023 12:46:59 - INFO - 	 missing_keys: []
01/18/2023 12:46:59 - INFO - 	 unexpected_keys: []
01/18/2023 12:46:59 - INFO - 	 mismatched_keys: []
01/18/2023 12:46:59 - INFO - 	 error_msgs: []
01/18/2023 12:46:59 - INFO - 	 Model Parameters: 590.0M, Transformer: 434.6M, Coref head: 155.4M
01/18/2023 12:46:59 - INFO - 	 Tokenize 1 inputs...


  0%|          | 0/1 [00:00<?, ?ba/s]

01/18/2023 12:47:00 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

"Alice goes down the rabbit hole. Where Alice would discover a new reality beyond Alice's expectations."

In [25]:
text="Alice goes down the rabbit hole. Where she would discover a new reality beyond her expectations."
doc = nlp(text,component_cfg={"fastcoref": {'resolve_text': True}})
doc._.resolved_text

01/18/2023 12:56:16 - INFO - 	 Tokenize 1 inputs...


  0%|          | 0/1 [00:00<?, ?ba/s]

01/18/2023 12:56:16 - INFO - 	 ***** Running Inference on 1 texts *****


Inference:   0%|          | 0/1 [00:00<?, ?it/s]

"Alice goes down the rabbit hole. Where Alice would discover a new reality beyond Alice's expectations."